# Futbol De Cabeza - kernels en GPU (Colab)

Corre los kernels de la catedra (`kernels.cu`) contra la version numpy y
compara los tiempos. **Runtime > Change runtime type > T4 GPU**.

El ZIP (`colab_pack.zip`, menos de 10 MB) se arma en la PC con
`py tools\pack_colab.py` y se sube en la segunda celda.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import files
subidos = files.upload()   # elegí colab_pack.zip
print(list(subidos))

In [ ]:
!unzip -o -q colab_pack.zip -d futbolDeCabeza
!pip install -q pycuda numpy
%cd futbolDeCabeza
!ls

In [ ]:
# Los kernels y la version numpy de las mismas operaciones, lado a lado
!python tools/bench_gpu.py --reps 10 --resolutions 426x240,854x480,1280x720,1920x1080

In [ ]:
import glob
import pandas as pd

csv = sorted(glob.glob('mediciones/bench_*.csv'))[-1]
datos = pd.read_csv(csv)
tabla = datos.pivot_table(index=['kernel', 'resolution'], columns='backend',
                          values='ms')
tabla['speedup'] = tabla['cpu'] / tabla['gpu']
print(csv)
tabla.round(2)

## Como leer los numeros

- `gpu` es el kernel de CUDA (un hilo por pixel, indice 2D) y `cpu` es la
  misma cuenta con numpy.
- `speedup` = ms de numpy / ms de la GPU: cuanto mas grande, mejor.
- Las imagenes aleatorias son siempre la misma (semilla fija), asi que dos
  corridas se pueden comparar.
- Si una resolucion no entra en la memoria de la T4, bajala en la celda 4.

In [ ]:
from google.colab import files
files.download(csv)